# 03 Long-Horizon Qualifying 2018-2025

Scalony notebook dla rozszerzenia projektu do lat 2018-2025: uzupelnienie danych historycznych, selekcja kierowcow, filtracja, ekstrakcja telemetrii, cechy i modele.


## Source: `18_backfill_qualifying_2018_2022.py`


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import time
import traceback

import fastf1
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "f1_cache"
EXPORT_DIR = BASE_DIR / "exports"
YEARS = [2018, 2019, 2020, 2021, 2022]
SESSION_NAME = "Q"
RATE_LIMIT_SLEEP_SECONDS = 3600
PROGRESS_PATH = EXPORT_DIR / "backfill_2018_2022_progress.csv"
SCHEDULE_PATH = EXPORT_DIR / "event_schedule_audit_2018_2022.csv"
AUDIT_PATH = EXPORT_DIR / "qualifying_session_audit_2018_2022.csv"
SUMMARY_PATH = EXPORT_DIR / "qualifying_session_audit_2018_2022_summary_by_season.csv"
LOG_PATH = EXPORT_DIR / "backfill_2018_2022.log"


def log(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def timed_to_seconds(series: pd.Series) -> pd.Series:
    td = pd.to_timedelta(series, errors="coerce")
    return td.dt.total_seconds()


def is_rate_limit_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    tokens = [
        "429",
        "rate limit",
        "ratelimit",
        "too many requests",
    ]
    return any(token in text for token in tokens)


def call_with_rate_limit_retry(fn, description: str):
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001
            if is_rate_limit_error(exc):
                log(
                    f"{description}: detected rate limit on attempt {attempt}. "
                    f"Sleeping for {RATE_LIMIT_SLEEP_SECONDS / 3600:.0f} hour(s) before retry."
                )
                time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                continue
            raise


def load_progress() -> pd.DataFrame:
    if PROGRESS_PATH.exists():
        return pd.read_csv(PROGRESS_PATH)
    return pd.DataFrame(
        columns=[
            "season",
            "round",
            "event_name",
            "status",
            "n_laps_rows",
            "n_unique_drivers_in_laps",
            "error_type",
            "error_message",
            "updated_at",
        ]
    )


def save_progress(df: pd.DataFrame) -> None:
    df.sort_values(["season", "round"], inplace=True)
    df.to_csv(PROGRESS_PATH, index=False)


def upsert_progress_row(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    key_mask = (
        (progress_df["season"] == row["season"]) &
        (progress_df["round"] == row["round"])
    )
    progress_df = progress_df.loc[~key_mask].copy()
    progress_df = pd.concat([progress_df, pd.DataFrame([row])], ignore_index=True)
    return progress_df


def build_schedule() -> pd.DataFrame:
    schedule_rows = []
    for year in YEARS:
        log(f"Loading event schedule for {year}")
        schedule = call_with_rate_limit_retry(
            lambda year=year: fastf1.get_event_schedule(year, include_testing=False),
            f"get_event_schedule({year})",
        )
        schedule = schedule.copy()
        if "RoundNumber" in schedule.columns:
            schedule = schedule[schedule["RoundNumber"].fillna(0).astype(int) > 0].copy()
        schedule["Season"] = year
        schedule_rows.append(schedule)
    schedule_df = pd.concat(schedule_rows, ignore_index=True)
    schedule_df.to_csv(SCHEDULE_PATH, index=False)
    return schedule_df


def session_audit_row(year: int, round_number: int, event_name: str) -> dict:
    session = fastf1.get_session(year, round_number, SESSION_NAME)
    call_with_rate_limit_retry(
        lambda: session.load(laps=True, telemetry=False, weather=True, messages=False),
        f"session.load({year}, R{round_number:02d}, {SESSION_NAME})",
    )

    laps = session.laps.copy()
    results = session.results.copy() if session.results is not None else pd.DataFrame()

    return {
        "season": year,
        "round": round_number,
        "event_name": event_name,
        "session_name": getattr(session, "name", None),
        "session_date": getattr(session, "date", None),
        "country": getattr(session.event, "Country", None) if hasattr(session, "event") else None,
        "location": getattr(session.event, "Location", None) if hasattr(session, "event") else None,
        "official_event_name": getattr(session.event, "OfficialEventName", None) if hasattr(session, "event") else None,
        "n_results_rows": len(results),
        "n_laps_rows": len(laps),
        "n_unique_drivers_in_laps": laps["Driver"].nunique() if "Driver" in laps.columns else np.nan,
        "lap_columns_count": len(laps.columns),
        "results_columns_count": len(results.columns),
        "has_compound": "Compound" in laps.columns,
        "has_tyre_life": "TyreLife" in laps.columns,
        "has_track_status": "TrackStatus" in laps.columns,
        "has_is_personal_best": "IsPersonalBest" in laps.columns,
        "has_is_accurate": "IsAccurate" in laps.columns,
        "sample_lap_time_seconds_mean": timed_to_seconds(laps["LapTime"]).mean() if "LapTime" in laps.columns else np.nan,
        "status": "ok",
    }


def main() -> None:
    CACHE_DIR.mkdir(exist_ok=True)
    EXPORT_DIR.mkdir(exist_ok=True)
    fastf1.Cache.enable_cache(str(CACHE_DIR))

    log("Starting 2018-2022 qualifying backfill")
    schedule_df = build_schedule()
    progress_df = load_progress()

    done_keys = {
        (int(row["season"]), int(row["round"]))
        for _, row in progress_df.iterrows()
        if row.get("status") == "ok"
    }

    audit_rows = []
    if AUDIT_PATH.exists():
        existing_audit_df = pd.read_csv(AUDIT_PATH)
        audit_rows.extend(existing_audit_df.to_dict("records"))

    for _, row in schedule_df.iterrows():
        year = int(row["Season"])
        round_number = int(row["RoundNumber"])
        event_name = row.get("EventName", f"Round {round_number}")

        if (year, round_number) in done_keys:
            log(f"Skipping already completed session: {year} R{round_number:02d} {event_name}")
            continue

        log(f"Processing {year} R{round_number:02d} {event_name}")
        try:
            audit_row = session_audit_row(year, round_number, event_name)
            audit_rows = [r for r in audit_rows if not (int(r["season"]) == year and int(r["round"]) == round_number)]
            audit_rows.append(audit_row)
            pd.DataFrame(audit_rows).sort_values(["season", "round"]).to_csv(AUDIT_PATH, index=False)

            progress_df = upsert_progress_row(
                progress_df,
                {
                    "season": year,
                    "round": round_number,
                    "event_name": event_name,
                    "status": "ok",
                    "n_laps_rows": audit_row["n_laps_rows"],
                    "n_unique_drivers_in_laps": audit_row["n_unique_drivers_in_laps"],
                    "error_type": None,
                    "error_message": None,
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                },
            )
            save_progress(progress_df)
            done_keys.add((year, round_number))
            log(f"Completed {year} R{round_number:02d} {event_name} with {audit_row['n_laps_rows']} laps")
        except Exception as exc:  # noqa: BLE001
            log(f"Failed {year} R{round_number:02d} {event_name}: {type(exc).__name__}: {exc}")
            trace = traceback.format_exc(limit=3)
            log(trace.strip())
            progress_df = upsert_progress_row(
                progress_df,
                {
                    "season": year,
                    "round": round_number,
                    "event_name": event_name,
                    "status": "failed",
                    "n_laps_rows": np.nan,
                    "n_unique_drivers_in_laps": np.nan,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                },
            )
            save_progress(progress_df)
            continue

    if AUDIT_PATH.exists():
        audit_df = pd.read_csv(AUDIT_PATH)
        summary_df = (
            audit_df.groupby("season", dropna=False)
            .agg(
                n_sessions=("round", "count"),
                n_laps_rows_total=("n_laps_rows", "sum"),
                n_laps_rows_mean=("n_laps_rows", "mean"),
                n_unique_drivers_mean=("n_unique_drivers_in_laps", "mean"),
            )
            .reset_index()
        )
        summary_df.to_csv(SUMMARY_PATH, index=False)
        log(f"Wrote season summary to {SUMMARY_PATH}")

    log("Backfill finished")


if __name__ == "__main__":
    main()


## Source: `19_backfill_qr_2018_2022.py`


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import time
import traceback

import fastf1
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
CACHE_DIR = BASE_DIR / "f1_cache"
EXPORT_DIR = BASE_DIR / "exports"
RAW_LAPS_DIR = EXPORT_DIR / "backfill_qr_raw_laps"
RAW_RESULTS_DIR = EXPORT_DIR / "backfill_qr_results"

YEARS = [2018, 2019, 2020, 2021, 2022]
SESSION_NAMES = ["Q", "R"]
RATE_LIMIT_SLEEP_SECONDS = 3600

PROGRESS_PATH = EXPORT_DIR / "backfill_qr_2018_2022_progress.csv"
SCHEDULE_PATH = EXPORT_DIR / "event_schedule_audit_2018_2022_qr.csv"
AUDIT_PATH = EXPORT_DIR / "session_audit_2018_2022_qr.csv"
SUMMARY_PATH = EXPORT_DIR / "session_audit_2018_2022_qr_summary.csv"
LOG_PATH = EXPORT_DIR / "backfill_qr_2018_2022.log"

WANTED_LAP_COLS = [
    "Time", "Driver", "DriverNumber", "LapTime", "LapNumber", "Stint",
    "PitOutTime", "PitInTime",
    "Sector1Time", "Sector2Time", "Sector3Time",
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
    "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
    "IsPersonalBest", "Compound", "TyreLife", "FreshTyre",
    "Team", "TrackStatus", "Position", "Deleted", "DeletedReason",
    "FastF1Generated", "IsAccurate"
]


def log(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    try:
        print(line, flush=True)
    except UnicodeEncodeError:
        safe_line = line.encode("ascii", errors="replace").decode("ascii")
        print(safe_line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def timed_to_seconds(series: pd.Series) -> pd.Series:
    td = pd.to_timedelta(series, errors="coerce")
    return td.dt.total_seconds()


def is_rate_limit_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(token in text for token in ["429", "rate limit", "ratelimit", "too many requests"])


def call_with_rate_limit_retry(fn, description: str):
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001
            if is_rate_limit_error(exc):
                log(
                    f"{description}: detected rate limit on attempt {attempt}. "
                    f"Sleeping for {RATE_LIMIT_SLEEP_SECONDS // 3600} hour before retry."
                )
                time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                continue
            raise


def load_progress() -> pd.DataFrame:
    if PROGRESS_PATH.exists():
        return pd.read_csv(PROGRESS_PATH)
    return pd.DataFrame(
        columns=[
            "season", "round", "session_name", "event_name", "status",
            "n_laps_rows", "n_unique_drivers_in_laps", "error_type",
            "error_message", "updated_at"
        ]
    )


def save_progress(df: pd.DataFrame) -> None:
    df.sort_values(["season", "round", "session_name"], inplace=True)
    df.to_csv(PROGRESS_PATH, index=False)


def upsert_progress_row(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    key_mask = (
        (progress_df["season"] == row["season"]) &
        (progress_df["round"] == row["round"]) &
        (progress_df["session_name"] == row["session_name"])
    )
    progress_df = progress_df.loc[~key_mask].copy()
    progress_df = pd.concat([progress_df, pd.DataFrame([row])], ignore_index=True)
    return progress_df


def build_schedule() -> pd.DataFrame:
    schedule_rows = []
    for year in YEARS:
        log(f"Loading event schedule for {year}")
        schedule = call_with_rate_limit_retry(
            lambda year=year: fastf1.get_event_schedule(year, include_testing=False),
            f"get_event_schedule({year})",
        )
        schedule = schedule.copy()
        if "RoundNumber" in schedule.columns:
            schedule = schedule[schedule["RoundNumber"].fillna(0).astype(int) > 0].copy()
        schedule["Season"] = year
        schedule_rows.append(schedule)
    schedule_df = pd.concat(schedule_rows, ignore_index=True)
    schedule_df.to_csv(SCHEDULE_PATH, index=False)
    return schedule_df


def session_output_paths(year: int, round_number: int, session_name: str) -> tuple[Path, Path]:
    stem = f"{year}_R{round_number:02d}_{session_name}"
    return RAW_LAPS_DIR / f"{stem}_laps.csv", RAW_RESULTS_DIR / f"{stem}_results.csv"


def fetch_session(year: int, round_number: int, event_name: str, session_name: str) -> dict:
    session = fastf1.get_session(year, round_number, session_name)
    call_with_rate_limit_retry(
        lambda: session.load(laps=True, telemetry=False, weather=True, messages=False),
        f"session.load({year}, R{round_number:02d}, {session_name})",
    )

    laps = session.laps.copy()
    results = session.results.copy() if session.results is not None else pd.DataFrame()

    existing_cols = [c for c in WANTED_LAP_COLS if c in laps.columns]
    laps = laps[existing_cols].copy()
    laps["season"] = year
    laps["round"] = round_number
    laps["event_name"] = event_name
    laps["session_name"] = session_name
    laps["LapTimeSeconds"] = timed_to_seconds(laps["LapTime"]) if "LapTime" in laps.columns else np.nan
    laps["Sector1TimeSeconds"] = timed_to_seconds(laps["Sector1Time"]) if "Sector1Time" in laps.columns else np.nan
    laps["Sector2TimeSeconds"] = timed_to_seconds(laps["Sector2Time"]) if "Sector2Time" in laps.columns else np.nan
    laps["Sector3TimeSeconds"] = timed_to_seconds(laps["Sector3Time"]) if "Sector3Time" in laps.columns else np.nan

    if not results.empty:
        results = results.copy()
        results["season"] = year
        results["round"] = round_number
        results["event_name"] = event_name
        results["session_name"] = session_name

    laps_path, results_path = session_output_paths(year, round_number, session_name)
    laps.to_csv(laps_path, index=False)
    results.to_csv(results_path, index=False)

    return {
        "season": year,
        "round": round_number,
        "event_name": event_name,
        "session_name": session_name,
        "session_date": getattr(session, "date", None),
        "country": getattr(session.event, "Country", None) if hasattr(session, "event") else None,
        "location": getattr(session.event, "Location", None) if hasattr(session, "event") else None,
        "official_event_name": getattr(session.event, "OfficialEventName", None) if hasattr(session, "event") else None,
        "n_results_rows": len(results),
        "n_laps_rows": len(laps),
        "n_unique_drivers_in_laps": laps["Driver"].nunique() if "Driver" in laps.columns else np.nan,
        "lap_columns_count": len(laps.columns),
        "results_columns_count": len(results.columns),
        "has_compound": "Compound" in laps.columns,
        "has_tyre_life": "TyreLife" in laps.columns,
        "has_track_status": "TrackStatus" in laps.columns,
        "has_is_personal_best": "IsPersonalBest" in laps.columns,
        "has_is_accurate": "IsAccurate" in laps.columns,
        "sample_lap_time_seconds_mean": laps["LapTimeSeconds"].mean() if "LapTimeSeconds" in laps.columns else np.nan,
        "laps_export_path": str(laps_path),
        "results_export_path": str(results_path),
        "status": "ok",
    }


def main() -> None:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")

    CACHE_DIR.mkdir(exist_ok=True)
    EXPORT_DIR.mkdir(exist_ok=True)
    RAW_LAPS_DIR.mkdir(exist_ok=True)
    RAW_RESULTS_DIR.mkdir(exist_ok=True)
    fastf1.Cache.enable_cache(str(CACHE_DIR))

    log("Starting 2018-2022 qualifying/race backfill")
    schedule_df = build_schedule()
    progress_df = load_progress()

    done_keys = {
        (int(row["season"]), int(row["round"]), str(row["session_name"]))
        for _, row in progress_df.iterrows()
        if row.get("status") == "ok"
    }

    audit_rows = []
    if AUDIT_PATH.exists():
        existing_audit_df = pd.read_csv(AUDIT_PATH)
        audit_rows.extend(existing_audit_df.to_dict("records"))

    for _, row in schedule_df.iterrows():
        year = int(row["Season"])
        round_number = int(row["RoundNumber"])
        event_name = row.get("EventName", f"Round {round_number}")

        for session_name in SESSION_NAMES:
            if (year, round_number, session_name) in done_keys:
                log(f"Skipping already completed session: {year} R{round_number:02d} {session_name} {event_name}")
                continue

            log(f"Processing {year} R{round_number:02d} {session_name} {event_name}")
            try:
                audit_row = fetch_session(year, round_number, event_name, session_name)
                audit_rows = [
                    r for r in audit_rows
                    if not (
                        int(r["season"]) == year and
                        int(r["round"]) == round_number and
                        str(r["session_name"]) == session_name
                    )
                ]
                audit_rows.append(audit_row)
                pd.DataFrame(audit_rows).sort_values(["season", "round", "session_name"]).to_csv(AUDIT_PATH, index=False)

                progress_df = upsert_progress_row(
                    progress_df,
                    {
                        "season": year,
                        "round": round_number,
                        "session_name": session_name,
                        "event_name": event_name,
                        "status": "ok",
                        "n_laps_rows": audit_row["n_laps_rows"],
                        "n_unique_drivers_in_laps": audit_row["n_unique_drivers_in_laps"],
                        "error_type": None,
                        "error_message": None,
                        "updated_at": datetime.now().isoformat(timespec="seconds"),
                    },
                )
                save_progress(progress_df)
                done_keys.add((year, round_number, session_name))
                log(f"Completed {year} R{round_number:02d} {session_name} {event_name} with {audit_row['n_laps_rows']} laps")
            except Exception as exc:  # noqa: BLE001
                log(f"Failed {year} R{round_number:02d} {session_name} {event_name}: {type(exc).__name__}: {exc}")
                trace = traceback.format_exc(limit=3)
                log(trace.strip())
                progress_df = upsert_progress_row(
                    progress_df,
                    {
                        "season": year,
                        "round": round_number,
                        "session_name": session_name,
                        "event_name": event_name,
                        "status": "failed",
                        "n_laps_rows": np.nan,
                        "n_unique_drivers_in_laps": np.nan,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                        "updated_at": datetime.now().isoformat(timespec="seconds"),
                    },
                )
                save_progress(progress_df)
                continue

    if AUDIT_PATH.exists():
        audit_df = pd.read_csv(AUDIT_PATH)
        summary_df = (
            audit_df.groupby(["season", "session_name"], dropna=False)
            .agg(
                n_sessions=("round", "count"),
                n_laps_rows_total=("n_laps_rows", "sum"),
                n_laps_rows_mean=("n_laps_rows", "mean"),
                n_unique_drivers_mean=("n_unique_drivers_in_laps", "mean"),
            )
            .reset_index()
        )
        summary_df.to_csv(SUMMARY_PATH, index=False)
        log(f"Wrote session summary to {SUMMARY_PATH}")

    log("Qualifying/race backfill finished")


if __name__ == "__main__":
    main()


## Source: `20_historical_qr_driver_audit.py`


In [ ]:
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("exports")
RAW_LAPS_DIR = EXPORT_DIR / "backfill_qr_raw_laps"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def load_raw_laps() -> pd.DataFrame:
    frames = []
    for path in sorted(RAW_LAPS_DIR.glob("*.csv")):
        try:
            df = pd.read_csv(path, usecols=["season", "round", "session_name", "Driver", "Team", "LapTimeSeconds"])
        except Exception:
            continue
        if df.empty:
            continue
        df["source_file"] = path.name
        frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["season", "round", "session_name", "Driver", "Team", "LapTimeSeconds", "source_file"])
    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=["season", "round", "session_name", "Driver"])
    out["season"] = out["season"].astype(int)
    out["round"] = out["round"].astype(int)
    out["session_key"] = (
        out["season"].astype(str) + "_" +
        out["round"].astype(str) + "_" +
        out["session_name"].astype(str)
    )
    return out


def main() -> None:
    laps_df = load_raw_laps()
    export_csv(laps_df, "historical_qr_loaded_laps_snapshot")

    if laps_df.empty:
        print("No raw lap files available yet.")
        return

    driver_coverage_df = (
        laps_df.groupby("Driver", dropna=False)
        .agg(
            n_lap_rows=("Driver", "size"),
            n_sessions=("session_key", "nunique"),
            n_seasons=("season", "nunique"),
            n_q_sessions=("session_name", lambda s: (s == "Q").sum()),
            n_r_sessions=("session_name", lambda s: (s == "R").sum()),
        )
        .reset_index()
        .sort_values(["n_seasons", "n_sessions", "n_lap_rows", "Driver"], ascending=[False, False, False, True])
    )
    export_csv(driver_coverage_df, "historical_qr_driver_coverage")

    season_presence_df = (
        laps_df.groupby(["Driver", "season"], dropna=False)
        .agg(
            n_lap_rows=("Driver", "size"),
            n_sessions=("session_key", "nunique"),
            n_q_rows=("session_name", lambda s: (s == "Q").sum()),
            n_r_rows=("session_name", lambda s: (s == "R").sum()),
        )
        .reset_index()
        .sort_values(["Driver", "season"])
    )
    export_csv(season_presence_df, "historical_qr_driver_season_presence")

    driver_season_matrix_df = (
        season_presence_df.pivot(index="Driver", columns="season", values="n_sessions")
        .fillna(0)
        .astype(int)
        .reset_index()
    )
    export_csv(driver_season_matrix_df, "historical_qr_driver_season_matrix")

    team_mix_df = (
        laps_df.dropna(subset=["Team"])
        .groupby(["Driver", "Team"], dropna=False)
        .agg(n_lap_rows=("Driver", "size"), n_sessions=("session_key", "nunique"))
        .reset_index()
        .sort_values(["Driver", "n_lap_rows"], ascending=[True, False])
    )
    export_csv(team_mix_df, "historical_qr_driver_team_mix")

    session_summary_df = (
        laps_df.groupby(["season", "session_name"], dropna=False)
        .agg(
            n_sessions=("session_key", "nunique"),
            n_lap_rows=("Driver", "size"),
            n_unique_drivers=("Driver", "nunique"),
        )
        .reset_index()
        .sort_values(["season", "session_name"])
    )
    export_csv(session_summary_df, "historical_qr_loaded_session_summary")

    print("Historical QR driver audit exports created.")
    print(driver_coverage_df.head(15).to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `21_historical_driver_set_candidates.py`


In [ ]:
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("exports")


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


coverage_df = load_csv("historical_qr_driver_coverage")
season_matrix_df = load_csv("historical_qr_driver_season_matrix")


season_cols = [c for c in season_matrix_df.columns if c != "Driver"]
matrix_long_df = season_matrix_df.melt(id_vars=["Driver"], var_name="season", value_name="n_sessions")
matrix_long_df["season"] = matrix_long_df["season"].astype(int)


coverage_rank_df = coverage_df.copy()
coverage_rank_df["avg_sessions_per_season"] = coverage_rank_df["n_sessions"] / coverage_rank_df["n_seasons"]
coverage_rank_df["avg_lap_rows_per_session"] = coverage_rank_df["n_lap_rows"] / coverage_rank_df["n_sessions"]
coverage_rank_df = coverage_rank_df.sort_values(
    ["n_seasons", "n_sessions", "n_lap_rows", "Driver"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
export_csv(coverage_rank_df, "historical_driver_coverage_ranked")


def build_candidate(name: str, drivers: list[str]) -> list[dict]:
    rows = []
    subset_cov = coverage_rank_df[coverage_rank_df["Driver"].isin(drivers)].copy()
    subset_sessions = matrix_long_df[matrix_long_df["Driver"].isin(drivers)].copy()
    seasons_all_present = (
        subset_sessions.groupby("season")["Driver"].nunique().reset_index(name="n_drivers_present")
    )
    seasons_all_present["all_present"] = seasons_all_present["n_drivers_present"] == len(drivers)
    rows.append(
        {
            "candidate_name": name,
            "drivers": ", ".join(drivers),
            "n_drivers": len(drivers),
            "min_n_seasons": int(subset_cov["n_seasons"].min()),
            "min_n_sessions": int(subset_cov["n_sessions"].min()),
            "mean_n_sessions": subset_cov["n_sessions"].mean(),
            "min_n_lap_rows": int(subset_cov["n_lap_rows"].min()),
            "mean_n_lap_rows": subset_cov["n_lap_rows"].mean(),
            "n_seasons_with_full_overlap": int(seasons_all_present["all_present"].sum()),
            "full_overlap_seasons": ", ".join(str(int(s)) for s in seasons_all_present.loc[seasons_all_present["all_present"], "season"]),
        }
    )
    return rows


season5_drivers = coverage_rank_df[coverage_rank_df["n_seasons"] == 5]["Driver"].tolist()
season4plus_drivers = coverage_rank_df[coverage_rank_df["n_seasons"] >= 4]["Driver"].tolist()

top5_season5 = season5_drivers[:5]
top6_season5 = season5_drivers[:6]
top8_season5 = season5_drivers[:8]
top10_season5 = season5_drivers[:10]
top8_season4plus = season4plus_drivers[:8]
top10_season4plus = season4plus_drivers[:10]
top12_season4plus = season4plus_drivers[:12]

candidate_rows = []
for name, drivers in [
    ("season5_top5", top5_season5),
    ("season5_top6", top6_season5),
    ("season5_top8", top8_season5),
    ("season5_top10", top10_season5),
    ("season4plus_top8", top8_season4plus),
    ("season4plus_top10", top10_season4plus),
    ("season4plus_top12", top12_season4plus),
]:
    candidate_rows.extend(build_candidate(name, drivers))

candidate_df = pd.DataFrame(candidate_rows).sort_values(
    ["n_seasons_with_full_overlap", "min_n_sessions", "min_n_lap_rows", "n_drivers"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
export_csv(candidate_df, "historical_driver_set_candidates")


candidate_members_df = pd.DataFrame(
    [
        {"candidate_name": row["candidate_name"], "driver": driver}
        for _, row in candidate_df.iterrows()
        for driver in row["drivers"].split(", ")
    ]
)
export_csv(candidate_members_df, "historical_driver_set_candidate_members")


best_candidate_name = candidate_df.iloc[0]["candidate_name"]
best_candidate_drivers = candidate_df.iloc[0]["drivers"]
best_candidate_summary_df = candidate_df.head(1).copy()
best_candidate_summary_df["recommendation_reason"] = (
    "Best current balance of long-horizon overlap and sample volume in the downloaded 2018-2022 snapshot."
)
export_csv(best_candidate_summary_df, "historical_driver_set_recommendation_snapshot")

print("Historical driver set candidates created.")
print(candidate_df.to_string(index=False))


## Source: `22_build_2018_2025_driver_selection_pool.py`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


EXPORT_DIR = Path("exports")
HIST_RAW_DIR = EXPORT_DIR / "backfill_qr_raw_laps"
CURRENT_Q_PATH = EXPORT_DIR / "all_qualifying_laps_raw.csv"
OPTIONAL_CURRENT_QR_DIR = EXPORT_DIR / "backfill_qr_raw_laps_2023_2025"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def standardize_raw_laps(df: pd.DataFrame, source_label: str) -> pd.DataFrame:
    out = df.copy()
    required = ["season", "round", "session_name", "Driver"]
    for col in required:
        if col not in out.columns:
            out[col] = np.nan
    if "LapTimeSeconds" not in out.columns:
        if "LapTime" in out.columns:
            out["LapTimeSeconds"] = pd.to_timedelta(out["LapTime"], errors="coerce").dt.total_seconds()
        else:
            out["LapTimeSeconds"] = np.nan
    if "Team" not in out.columns:
        out["Team"] = np.nan
    out["season"] = pd.to_numeric(out["season"], errors="coerce")
    out["round"] = pd.to_numeric(out["round"], errors="coerce")
    out = out.dropna(subset=["season", "round", "session_name", "Driver"]).copy()
    out["season"] = out["season"].astype(int)
    out["round"] = out["round"].astype(int)
    out["session_name"] = out["session_name"].astype(str)
    out["Driver"] = out["Driver"].astype(str)
    out["source_label"] = source_label
    out["session_key"] = (
        out["season"].astype(str) + "_" +
        out["round"].astype(str) + "_" +
        out["session_name"]
    )
    return out[["season", "round", "session_name", "session_key", "Driver", "Team", "LapTimeSeconds", "source_label"]]


def load_hist_2018_2022_qr() -> pd.DataFrame:
    frames = []
    for path in sorted(HIST_RAW_DIR.glob("*.csv")):
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if df.empty:
            continue
        frames.append(standardize_raw_laps(df, "2018_2022_qr_backfill"))
    if not frames:
        return pd.DataFrame(columns=["season", "round", "session_name", "session_key", "Driver", "Team", "LapTimeSeconds", "source_label"])
    return pd.concat(frames, ignore_index=True)


def load_current_2023_2025_q() -> pd.DataFrame:
    if not CURRENT_Q_PATH.exists():
        return pd.DataFrame(columns=["season", "round", "session_name", "session_key", "Driver", "Team", "LapTimeSeconds", "source_label"])
    df = pd.read_csv(CURRENT_Q_PATH)
    return standardize_raw_laps(df, "2023_2025_q_existing")


def load_optional_current_qr() -> pd.DataFrame:
    frames = []
    if not OPTIONAL_CURRENT_QR_DIR.exists():
        return pd.DataFrame(columns=["season", "round", "session_name", "session_key", "Driver", "Team", "LapTimeSeconds", "source_label"])
    for path in sorted(OPTIONAL_CURRENT_QR_DIR.glob("*.csv")):
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if df.empty:
            continue
        frames.append(standardize_raw_laps(df, "2023_2025_qr_backfill"))
    if not frames:
        return pd.DataFrame(columns=["season", "round", "session_name", "session_key", "Driver", "Team", "LapTimeSeconds", "source_label"])
    return pd.concat(frames, ignore_index=True)


hist_df = load_hist_2018_2022_qr()
current_q_df = load_current_2023_2025_q()
current_qr_df = load_optional_current_qr()

combined_df = pd.concat([hist_df, current_q_df, current_qr_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=["season", "round", "session_name", "Driver", "LapTimeSeconds", "source_label"])
export_csv(combined_df, "driver_selection_pool_2018_2025_raw_laps")


source_summary_df = (
    combined_df.groupby(["source_label", "season", "session_name"], dropna=False)
    .agg(
        n_session_rows=("Driver", "size"),
        n_unique_sessions=("session_key", "nunique"),
        n_unique_drivers=("Driver", "nunique"),
    )
    .reset_index()
    .sort_values(["season", "session_name", "source_label"])
)
export_csv(source_summary_df, "driver_selection_pool_2018_2025_source_summary")


driver_total_coverage_df = (
    combined_df.groupby("Driver", dropna=False)
    .agg(
        n_lap_rows=("Driver", "size"),
        n_sessions=("session_key", "nunique"),
        n_seasons=("season", "nunique"),
        n_q_rows=("session_name", lambda s: (s == "Q").sum()),
        n_r_rows=("session_name", lambda s: (s == "R").sum()),
        n_q_sessions=("session_key", lambda s: s[s.str.endswith("_Q")].nunique()),
        n_r_sessions=("session_key", lambda s: s[s.str.endswith("_R")].nunique()),
    )
    .reset_index()
)
driver_total_coverage_df["avg_sessions_per_season"] = driver_total_coverage_df["n_sessions"] / driver_total_coverage_df["n_seasons"]
driver_total_coverage_df["coverage_rank_score"] = (
    driver_total_coverage_df["n_seasons"] * 10000 +
    driver_total_coverage_df["n_sessions"] * 10 +
    driver_total_coverage_df["n_lap_rows"] / 1000.0
)
driver_total_coverage_df = driver_total_coverage_df.sort_values(
    ["n_seasons", "n_sessions", "n_lap_rows", "Driver"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
export_csv(driver_total_coverage_df, "driver_selection_pool_2018_2025_total_coverage")


q_only_df = combined_df[combined_df["session_name"] == "Q"].copy()
driver_q_coverage_df = (
    q_only_df.groupby("Driver", dropna=False)
    .agg(
        n_q_lap_rows=("Driver", "size"),
        n_q_sessions=("session_key", "nunique"),
        n_q_seasons=("season", "nunique"),
    )
    .reset_index()
)
driver_q_coverage_df["q_coverage_rank_score"] = (
    driver_q_coverage_df["n_q_seasons"] * 10000 +
    driver_q_coverage_df["n_q_sessions"] * 10 +
    driver_q_coverage_df["n_q_lap_rows"] / 1000.0
)
driver_q_coverage_df = driver_q_coverage_df.sort_values(
    ["n_q_seasons", "n_q_sessions", "n_q_lap_rows", "Driver"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
export_csv(driver_q_coverage_df, "driver_selection_pool_2018_2025_q_coverage")


season_matrix_df = (
    combined_df.groupby(["Driver", "season"], dropna=False)
    .agg(
        n_sessions=("session_key", "nunique"),
        n_q_sessions=("session_key", lambda s: s[s.str.endswith("_Q")].nunique()),
        n_r_sessions=("session_key", lambda s: s[s.str.endswith("_R")].nunique()),
        n_lap_rows=("Driver", "size"),
    )
    .reset_index()
)
export_csv(season_matrix_df, "driver_selection_pool_2018_2025_driver_season_presence")


distinctiveness_df = None
distinctiveness_path = EXPORT_DIR / "subset_driver_recall_ranking.csv"
if distinctiveness_path.exists():
    distinctiveness_df = pd.read_csv(distinctiveness_path).rename(columns={"Driver": "driver", "recall": "current_recall_2023_2025"})
    if "driver" not in distinctiveness_df.columns:
        if "driver" not in distinctiveness_df.columns and "Driver" in pd.read_csv(distinctiveness_path, nrows=0).columns:
            distinctiveness_df = pd.read_csv(distinctiveness_path).rename(columns={"Driver": "driver"})

if distinctiveness_df is not None and "driver" in distinctiveness_df.columns:
    shortlist_df = driver_total_coverage_df.copy()
    shortlist_df = shortlist_df.rename(columns={"Driver": "driver"})
    shortlist_df = shortlist_df.merge(
        distinctiveness_df[["driver"] + [c for c in distinctiveness_df.columns if c != "driver"]],
        on="driver",
        how="left",
    )
    shortlist_df["has_current_distinctiveness_signal"] = shortlist_df["current_recall_2023_2025"].notna()
else:
    shortlist_df = driver_total_coverage_df.rename(columns={"Driver": "driver"}).copy()
    shortlist_df["has_current_distinctiveness_signal"] = False
export_csv(shortlist_df, "driver_selection_pool_2018_2025_shortlist_with_current_signal")


top_total_10 = driver_total_coverage_df.head(10)["Driver"].tolist()
top_q_10 = driver_q_coverage_df.head(10)["Driver"].tolist()
candidate_pool = sorted(set(top_total_10).union(set(top_q_10)))
candidate_pool_df = pd.DataFrame({"driver": candidate_pool})
export_csv(candidate_pool_df, "driver_selection_pool_2018_2025_candidate_pool")


metadata_df = pd.DataFrame(
    [
        {
            "has_hist_2018_2022_qr": not hist_df.empty,
            "has_current_2023_2025_q": not current_q_df.empty,
            "has_current_2023_2025_qr": not current_qr_df.empty,
            "n_total_rows": len(combined_df),
            "n_total_drivers": combined_df["Driver"].nunique(),
            "n_total_seasons": combined_df["season"].nunique(),
            "n_total_sessions": combined_df["session_key"].nunique(),
            "note": "If 2023-2025 race raw laps are absent, total coverage is provisional for R sessions in those seasons.",
        }
    ]
)
export_csv(metadata_df, "driver_selection_pool_2018_2025_metadata")

print("2018-2025 driver selection pool exports created.")
print(metadata_df.to_string(index=False))
print()
print(driver_total_coverage_df.head(15).to_string(index=False))


## Source: `23_select_2018_2025_provisional_top5.py`


In [ ]:
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("exports")


def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(EXPORT_DIR / f"{name}.csv")


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


q_cov_df = load_csv("driver_selection_pool_2018_2025_q_coverage")
shortlist_df = load_csv("driver_selection_pool_2018_2025_shortlist_with_current_signal")


rank_df = shortlist_df.merge(
    q_cov_df[["Driver", "n_q_sessions", "n_q_seasons", "q_coverage_rank_score"]],
    left_on="driver",
    right_on="Driver",
    how="left",
).drop(columns=["Driver"])

if "n_q_sessions_y" in rank_df.columns:
    rank_df["n_q_sessions_final"] = rank_df["n_q_sessions_y"].fillna(rank_df.get("n_q_sessions_x"))
else:
    rank_df["n_q_sessions_final"] = rank_df["n_q_sessions"]

rank_df = rank_df[rank_df["has_current_distinctiveness_signal"]].copy()
rank_df["coverage_rank"] = rank_df["q_coverage_rank_score"].rank(ascending=False, method="dense")
rank_df["distinctiveness_rank"] = rank_df["current_recall_2023_2025"].rank(ascending=False, method="dense")


def minmax(series: pd.Series) -> pd.Series:
    min_v = series.min()
    max_v = series.max()
    if max_v == min_v:
        return pd.Series(1.0, index=series.index)
    return (series - min_v) / (max_v - min_v)


rank_df["coverage_norm"] = minmax(rank_df["q_coverage_rank_score"])
rank_df["distinctiveness_norm"] = minmax(rank_df["current_recall_2023_2025"])
rank_df["balanced_score"] = 0.5 * rank_df["coverage_norm"] + 0.5 * rank_df["distinctiveness_norm"]

rank_df["coverage_floor_ge_7_seasons"] = rank_df["n_q_seasons"] >= 7
rank_df["coverage_floor_ge_8_seasons"] = rank_df["n_q_seasons"] >= 8
rank_df["coverage_floor_ge_140_q_sessions"] = rank_df["n_q_sessions_final"] >= 140

rank_df = rank_df.sort_values(
    ["balanced_score", "current_recall_2023_2025", "q_coverage_rank_score"],
    ascending=[False, False, False],
).reset_index(drop=True)
export_csv(rank_df, "driver_selection_2018_2025_provisional_ranked")


coverage_top5 = (
    rank_df.sort_values(["q_coverage_rank_score", "current_recall_2023_2025"], ascending=[False, False])
    .head(5)["driver"]
    .tolist()
)

distinctiveness_with_floor_top5 = (
    rank_df[rank_df["coverage_floor_ge_7_seasons"]]
    .sort_values(["current_recall_2023_2025", "q_coverage_rank_score"], ascending=[False, False])
    .head(5)["driver"]
    .tolist()
)

balanced_top5 = (
    rank_df[rank_df["coverage_floor_ge_7_seasons"]]
    .sort_values(["balanced_score", "current_recall_2023_2025", "q_coverage_rank_score"], ascending=[False, False, False])
    .head(5)["driver"]
    .tolist()
)

balanced_top6 = (
    rank_df[rank_df["coverage_floor_ge_7_seasons"]]
    .sort_values(["balanced_score", "current_recall_2023_2025", "q_coverage_rank_score"], ascending=[False, False, False])
    .head(6)["driver"]
    .tolist()
)

candidate_sets_df = pd.DataFrame(
    [
        {
            "candidate_name": "coverage_top5_q",
            "drivers": ", ".join(coverage_top5),
            "selection_logic": "Top 5 by 2018-2025 qualifying coverage among drivers with current 2023-2025 distinctiveness signal.",
        },
        {
            "candidate_name": "distinctiveness_top5_with_coverage_floor",
            "drivers": ", ".join(distinctiveness_with_floor_top5),
            "selection_logic": "Top 5 by current distinctiveness, restricted to drivers with at least 7 qualifying seasons in the 2018-2025 pool.",
        },
        {
            "candidate_name": "balanced_top5",
            "drivers": ", ".join(balanced_top5),
            "selection_logic": "Top 5 by equal-weight balance of 2018-2025 qualifying coverage and current 2023-2025 distinctiveness, with a 7-season coverage floor.",
        },
        {
            "candidate_name": "balanced_top6",
            "drivers": ", ".join(balanced_top6),
            "selection_logic": "Same as balanced_top5, but keeping 6 drivers for a slightly wider strict benchmark.",
        },
    ]
)
export_csv(candidate_sets_df, "driver_selection_2018_2025_provisional_top5_candidates")


recommended_name = "balanced_top5"
recommended_drivers = ", ".join(balanced_top5)
recommendation_df = pd.DataFrame(
    [
        {
            "recommended_candidate_name": recommended_name,
            "drivers": recommended_drivers,
            "reason": "Best current trade-off between long-horizon qualifying coverage and already-observed driver distinctiveness in the 2023-2025 benchmark.",
            "status": "provisional_until_2018_2022_backfill_finishes",
        }
    ]
)
export_csv(recommendation_df, "driver_selection_2018_2025_provisional_recommendation")

print("Provisional 2018-2025 top-5 selector exports created.")
print(candidate_sets_df.to_string(index=False))


## Source: `24_build_long_horizon_strict_qualifying.py`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


EXPORT_DIR = Path("exports")
BACKFILL_Q_DIR = EXPORT_DIR / "backfill_qr_raw_laps"
CURRENT_Q_PATH = EXPORT_DIR / "all_qualifying_laps_raw.csv"
CANDIDATE_PATH = EXPORT_DIR / "driver_selection_2018_2025_provisional_top5_candidates.csv"

SLICK_COMPOUNDS = {
    "SOFT", "MEDIUM", "HARD",
    "SUPERSOFT", "ULTRASOFT", "HYPERSOFT", "SUPERHARD",
    "C1", "C2", "C3", "C4", "C5",
    "XSOFT", "PRIME", "OPTION",
}
WET_COMPOUNDS = {"INTERMEDIATE", "WET"}


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def timed_to_seconds(series: pd.Series) -> pd.Series:
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()


def standardize_df(df: pd.DataFrame, source_label: str) -> pd.DataFrame:
    out = df.copy()
    wanted = [
        "Time", "Driver", "DriverNumber", "LapTime", "LapNumber", "Stint", "PitOutTime", "PitInTime",
        "Sector1Time", "Sector2Time", "Sector3Time",
        "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
        "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
        "IsPersonalBest", "Compound", "TyreLife", "FreshTyre",
        "Team", "TrackStatus", "Position", "Deleted", "DeletedReason",
        "FastF1Generated", "IsAccurate",
        "season", "round", "event_name", "session_name",
        "LapTimeSeconds", "Sector1TimeSeconds", "Sector2TimeSeconds", "Sector3TimeSeconds",
        "session_max_rainfall", "session_is_dry_by_weather",
    ]
    for col in wanted:
        if col not in out.columns:
            out[col] = np.nan

    if out["LapTimeSeconds"].isna().all() and "LapTime" in out.columns:
        out["LapTimeSeconds"] = timed_to_seconds(out["LapTime"])
    if out["Sector1TimeSeconds"].isna().all() and "Sector1Time" in out.columns:
        out["Sector1TimeSeconds"] = timed_to_seconds(out["Sector1Time"])
    if out["Sector2TimeSeconds"].isna().all() and "Sector2Time" in out.columns:
        out["Sector2TimeSeconds"] = timed_to_seconds(out["Sector2Time"])
    if out["Sector3TimeSeconds"].isna().all() and "Sector3Time" in out.columns:
        out["Sector3TimeSeconds"] = timed_to_seconds(out["Sector3Time"])

    out["season"] = pd.to_numeric(out["season"], errors="coerce")
    out["round"] = pd.to_numeric(out["round"], errors="coerce")
    out["LapNumber"] = pd.to_numeric(out["LapNumber"], errors="coerce")
    out["TyreLife"] = pd.to_numeric(out["TyreLife"], errors="coerce")

    out = out.dropna(subset=["season", "round", "Driver", "session_name"]).copy()
    out["season"] = out["season"].astype(int)
    out["round"] = out["round"].astype(int)
    out["session_name"] = out["session_name"].astype(str)
    out["Driver"] = out["Driver"].astype(str)
    out["event_name"] = out["event_name"].astype(str)
    out["source_label"] = source_label
    out["session_key"] = out["season"].astype(str) + "_" + out["round"].astype(str) + "_" + out["session_name"]

    for col in ["Deleted", "FastF1Generated", "IsAccurate", "IsPersonalBest", "FreshTyre", "session_is_dry_by_weather"]:
        if col in out.columns:
            out[col] = (
                out[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({"true": True, "false": False})
                .where(~out[col].isna(), np.nan)
            )

    return out[wanted + ["source_label", "session_key"]]


def load_backfill_q() -> pd.DataFrame:
    frames = []
    for path in sorted(BACKFILL_Q_DIR.glob("*_Q_laps.csv")):
        df = pd.read_csv(path, low_memory=False)
        if df.empty:
            continue
        frames.append(standardize_df(df, "2018_2022_q_backfill"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_current_q() -> pd.DataFrame:
    df = pd.read_csv(CURRENT_Q_PATH, low_memory=False)
    return standardize_df(df, "2023_2025_q_existing")


def infer_dry_session(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    comp = out["Compound"].astype(str).str.upper().str.strip()
    out["compound_upper"] = comp
    out["is_wet_compound"] = comp.isin(WET_COMPOUNDS)
    out["is_slick_compound"] = comp.isin(SLICK_COMPOUNDS)

    session_flags = (
        out.groupby("session_key", dropna=False)
        .agg(
            has_weather_dry=("session_is_dry_by_weather", lambda s: s.dropna().iloc[0] if s.dropna().size else np.nan),
            any_wet_compound=("is_wet_compound", "any"),
            any_slick_compound=("is_slick_compound", "any"),
            n_compounds=("compound_upper", lambda s: s.replace("NAN", np.nan).dropna().nunique()),
        )
        .reset_index()
    )

    def choose_flag(row):
        if pd.notna(row["has_weather_dry"]):
            return bool(row["has_weather_dry"]), "weather_flag"
        inferred = bool((not row["any_wet_compound"]) and row["any_slick_compound"])
        return inferred, "compound_inference"

    chosen = session_flags.apply(choose_flag, axis=1, result_type="expand")
    chosen.columns = ["session_is_dry_final", "dry_flag_source"]
    session_flags = pd.concat([session_flags, chosen], axis=1)
    out = out.merge(session_flags[["session_key", "session_is_dry_final", "dry_flag_source"]], on="session_key", how="left")
    return out


def summarize_step(df: pd.DataFrame, step: str) -> dict:
    return {
        "step": step,
        "n_rows": len(df),
        "n_unique_drivers": df["Driver"].nunique(),
        "n_unique_sessions": df["session_key"].nunique(),
    }


def choose_best_lap_per_driver_session(df: pd.DataFrame) -> pd.DataFrame:
    work = df.dropna(subset=["LapTimeSeconds"]).copy()
    work = work.sort_values(["Driver", "session_key", "LapTimeSeconds", "LapNumber"])
    return work.groupby(["Driver", "session_key"], as_index=False).head(1).reset_index(drop=True)


def add_lap_key(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["lap_key"] = (
        out["season"].astype(str) + "_r" + out["round"].astype(str).str.zfill(2) +
        "_" + out["session_name"] + "_" + out["Driver"] +
        "_lap" + out["LapNumber"].fillna(-1).astype(int).astype(str)
    )
    return out


def main() -> None:
    backfill_q_df = load_backfill_q()
    current_q_df = load_current_q()
    all_q_df = pd.concat([backfill_q_df, current_q_df], ignore_index=True)
    all_q_df = infer_dry_session(all_q_df)
    all_q_df = add_lap_key(all_q_df)
    export_csv(all_q_df, "all_qualifying_laps_raw_2018_2025")

    filter_rows = []
    filter_rows.append(summarize_step(all_q_df, "00_raw_all_qualifying_laps"))

    step_01 = all_q_df[all_q_df["LapTimeSeconds"].notna()].copy()
    filter_rows.append(summarize_step(step_01, "01_lap_time_not_null"))

    step_02 = step_01[step_01["session_is_dry_final"] == True].copy()
    filter_rows.append(summarize_step(step_02, "02_dry_sessions_only"))

    step_03 = step_02[step_02["IsAccurate"] == True].copy()
    filter_rows.append(summarize_step(step_03, "03_is_accurate_true"))

    if "Deleted" in step_03.columns:
        step_04 = step_03[(step_03["Deleted"] != True) | (step_03["Deleted"].isna())].copy()
    else:
        step_04 = step_03.copy()
    filter_rows.append(summarize_step(step_04, "04_not_deleted"))

    if "FastF1Generated" in step_04.columns:
        step_05 = step_04[(step_04["FastF1Generated"] != True) | (step_04["FastF1Generated"].isna())].copy()
    else:
        step_05 = step_04.copy()
    filter_rows.append(summarize_step(step_05, "05_not_fastf1_generated"))

    track_status_num = pd.to_numeric(step_05["TrackStatus"], errors="coerce")
    step_06 = step_05[track_status_num == 1].copy()
    filter_rows.append(summarize_step(step_06, "06_track_status_eq_1"))

    step_07a = step_06[step_06["IsPersonalBest"] == True].copy()
    filter_rows.append(summarize_step(step_07a, "07a_personal_best_only"))

    filter_summary_df = pd.DataFrame(filter_rows)
    export_csv(filter_summary_df, "lap_filter_summary_2018_2025_strict")

    clean_laps_all = step_06.copy().sort_values(["season", "round", "Driver", "LapNumber"])
    clean_laps_pb = step_07a.copy().sort_values(["season", "round", "Driver", "LapNumber"])
    session_driver_best = choose_best_lap_per_driver_session(step_06)

    export_csv(clean_laps_all, "clean_laps_all_2018_2025_strict")
    export_csv(clean_laps_pb, "clean_laps_personal_best_only_2018_2025_strict")
    export_csv(session_driver_best, "session_driver_best_laps_2018_2025_strict")

    season_summary_df = (
        session_driver_best.groupby("season", dropna=False)
        .agg(
            n_best_laps=("lap_key", "count"),
            n_drivers=("Driver", "nunique"),
            n_sessions=("session_key", "nunique"),
        )
        .reset_index()
        .sort_values("season")
    )
    export_csv(season_summary_df, "season_clean_summary_2018_2025_strict")

    driver_counts_df = (
        session_driver_best.groupby("Driver", dropna=False)
        .agg(n_laps=("lap_key", "count"), n_seasons=("season", "nunique"), n_sessions=("session_key", "nunique"))
        .reset_index()
        .sort_values(["n_laps", "n_seasons", "Driver"], ascending=[False, False, True])
    )
    export_csv(driver_counts_df, "driver_session_counts_clean_2018_2025_strict")

    dry_source_summary_df = (
        all_q_df.groupby(["season", "dry_flag_source", "session_is_dry_final"], dropna=False)
        .agg(n_sessions=("session_key", "nunique"))
        .reset_index()
        .sort_values(["season", "dry_flag_source", "session_is_dry_final"])
    )
    export_csv(dry_source_summary_df, "dry_session_flag_summary_2018_2025_strict")

    candidates_df = pd.read_csv(CANDIDATE_PATH)
    subset_lookup = dict(zip(candidates_df["candidate_name"], candidates_df["drivers"]))

    for subset_name in ["balanced_top5", "balanced_top6"]:
        if subset_name not in subset_lookup:
            continue
        drivers = [d.strip() for d in subset_lookup[subset_name].split(",")]
        subset_df = session_driver_best[session_driver_best["Driver"].isin(drivers)].copy()
        export_csv(subset_df, f"{subset_name}_session_driver_best_laps_2018_2025_strict")

        subset_counts_df = (
            subset_df.groupby("Driver", dropna=False)
            .agg(n_laps=("lap_key", "count"), n_seasons=("season", "nunique"), n_sessions=("session_key", "nunique"))
            .reset_index()
            .sort_values(["n_laps", "Driver"], ascending=[False, True])
        )
        export_csv(subset_counts_df, f"{subset_name}_class_balance_2018_2025_strict")

    metadata_df = pd.DataFrame(
        [
            {
                "strict_source_horizon": "2018-2025",
                "included_sessions": "Qualifying only",
                "dry_flag_note": "2018-2022 sessions use compound-based dry inference when weather-derived dry flag is unavailable; 2023-2025 uses existing weather-derived dry flag.",
                "recommended_top5": subset_lookup.get("balanced_top5"),
                "recommended_top6": subset_lookup.get("balanced_top6"),
            }
        ]
    )
    export_csv(metadata_df, "long_horizon_strict_qualifying_metadata")

    print("Built long-horizon strict qualifying datasets.")
    print(filter_summary_df.to_string(index=False))
    print()
    print(driver_counts_df.head(15).to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `25_extract_long_horizon_telemetry.py`


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import time
import traceback

import fastf1
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
EXPORT_DIR = BASE_DIR / "exports"
CACHE_DIR = BASE_DIR / "f1_cache"
TARGET_PATH = EXPORT_DIR / "balanced_top6_session_driver_best_laps_2018_2025_strict.csv"

AUDIT_PATH = EXPORT_DIR / "balanced_top6_telemetry_audit_2018_2025_strict.csv"
MERGED_PATH = EXPORT_DIR / "balanced_top6_telemetry_merged_2018_2025_strict.csv"
SUCCESS_PATH = EXPORT_DIR / "balanced_top6_telemetry_success_summary_2018_2025_strict.csv"
DRIVER_COUNTS_PATH = EXPORT_DIR / "balanced_top6_telemetry_driver_counts_2018_2025_strict.csv"
LOG_PATH = EXPORT_DIR / "balanced_top6_telemetry_extract_2018_2025_strict.log"

RATE_LIMIT_SLEEP_SECONDS = 3600


def log(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    try:
        print(line, flush=True)
    except UnicodeEncodeError:
        safe_line = line.encode("ascii", errors="replace").decode("ascii")
        print(safe_line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def timed_to_seconds(series: pd.Series) -> pd.Series:
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()


def is_rate_limit_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(token in text for token in ["429", "rate limit", "ratelimit", "too many requests"])


def call_with_rate_limit_retry(fn, description: str):
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001
            if is_rate_limit_error(exc):
                log(
                    f"{description}: detected rate limit on attempt {attempt}. "
                    f"Sleeping for {RATE_LIMIT_SLEEP_SECONDS // 3600} hour before retry."
                )
                time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                continue
            raise


def load_existing_done_lap_keys() -> set[str]:
    if not AUDIT_PATH.exists():
        return set()
    audit_df = pd.read_csv(AUDIT_PATH)
    return set(audit_df.loc[audit_df["status"] == "ok", "lap_key"].astype(str))


def append_csv_rows(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def append_df(path: Path, df: pd.DataFrame) -> None:
    if df.empty:
        return
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def main() -> None:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")

    fastf1.Cache.enable_cache(str(CACHE_DIR))

    targets_df = pd.read_csv(TARGET_PATH)
    if "lap_key" not in targets_df.columns:
        raise ValueError("Target file must include lap_key column.")

    done_lap_keys = load_existing_done_lap_keys()
    pending_df = targets_df[~targets_df["lap_key"].astype(str).isin(done_lap_keys)].copy()

    log(f"Telemetry extraction target laps: {len(targets_df)}; already done: {len(done_lap_keys)}; pending: {len(pending_df)}")
    if pending_df.empty:
        log("No pending laps. Building summaries only.")
    else:
        grouped = pending_df.groupby(["season", "round"], sort=True)

        for (season, round_number), group in grouped:
            log(f"Loading session {season} R{int(round_number):02d} Q with {len(group)} pending laps")
            telemetry_rows = []
            audit_rows = []

            try:
                session = fastf1.get_session(int(season), int(round_number), "Q")
                call_with_rate_limit_retry(
                    lambda: session.load(laps=True, telemetry=True, weather=False, messages=False),
                    f"session.load({season}, R{int(round_number):02d}, Q, telemetry=True)",
                )
                laps = session.laps.copy()

                for _, target in group.iterrows():
                    driver = target["Driver"]
                    lap_number = target["LapNumber"]
                    lap_key = str(target["lap_key"])

                    try:
                        candidate = laps[
                            (laps["Driver"] == driver) &
                            (laps["LapNumber"] == lap_number)
                        ].copy()

                        if len(candidate) != 1:
                            audit_rows.append(
                                {
                                    "lap_key": lap_key,
                                    "season": season,
                                    "round": round_number,
                                    "Driver": driver,
                                    "LapNumber": lap_number,
                                    "status": "lap_match_not_unique",
                                    "n_matches": len(candidate),
                                }
                            )
                            continue

                        lap = candidate.iloc[0]
                        car_data = lap.get_car_data().copy()
                        pos_data = lap.get_pos_data().copy()

                        if "Time" in car_data.columns:
                            car_data["telemetry_time_sec"] = timed_to_seconds(car_data["Time"])
                        if "SessionTime" in car_data.columns:
                            car_data["telemetry_session_time_sec"] = timed_to_seconds(car_data["SessionTime"])
                        if "Date" in car_data.columns:
                            car_data["telemetry_date"] = car_data["Date"].astype(str)

                        if "Time" in pos_data.columns:
                            pos_data["pos_time_sec"] = timed_to_seconds(pos_data["Time"])
                        if "SessionTime" in pos_data.columns:
                            pos_data["pos_session_time_sec"] = timed_to_seconds(pos_data["SessionTime"])
                        if "Date" in pos_data.columns:
                            pos_data["pos_date"] = pos_data["Date"].astype(str)

                        for df_ in [car_data, pos_data]:
                            df_["lap_key"] = lap_key
                            df_["season"] = season
                            df_["round"] = round_number
                            df_["Driver"] = driver
                            df_["LapNumber"] = lap_number

                        for col in [
                            "LapTimeSeconds", "Sector1TimeSeconds", "Sector2TimeSeconds", "Sector3TimeSeconds",
                            "Compound", "TyreLife", "Team", "session_key"
                        ]:
                            car_data[col] = target.get(col)

                        pos_keep = [
                            c for c in pos_data.columns if c in [
                                "lap_key", "X", "Y", "Z", "Status",
                                "pos_time_sec", "pos_session_time_sec", "pos_date"
                            ]
                        ]
                        pos_data = pos_data[pos_keep].copy()

                        car_data = car_data.reset_index(drop=True)
                        pos_data = pos_data.reset_index(drop=True)
                        n = min(len(car_data), len(pos_data))

                        merged = pd.concat(
                            [
                                car_data.iloc[:n].reset_index(drop=True),
                                pos_data.drop(columns=["lap_key"], errors="ignore").iloc[:n].reset_index(drop=True),
                            ],
                            axis=1,
                        )
                        merged["sample_idx"] = np.arange(len(merged))
                        telemetry_rows.append(merged)

                        audit_rows.append(
                            {
                                "lap_key": lap_key,
                                "season": season,
                                "round": round_number,
                                "Driver": driver,
                                "LapNumber": lap_number,
                                "status": "ok",
                                "car_rows": len(car_data),
                                "pos_rows": len(pos_data),
                                "merged_rows": len(merged),
                            }
                        )
                    except Exception as exc:  # noqa: BLE001
                        audit_rows.append(
                            {
                                "lap_key": lap_key,
                                "season": season,
                                "round": round_number,
                                "Driver": driver,
                                "LapNumber": lap_number,
                                "status": "lap_extract_error",
                                "error_type": type(exc).__name__,
                                "error_msg": str(exc),
                            }
                        )

                append_df(MERGED_PATH, pd.concat(telemetry_rows, ignore_index=True) if telemetry_rows else pd.DataFrame())
                append_csv_rows(AUDIT_PATH, audit_rows)
                ok_count = sum(1 for row in audit_rows if row["status"] == "ok")
                log(f"Completed {season} R{int(round_number):02d} Q: {ok_count}/{len(group)} laps ok")

            except Exception as exc:  # noqa: BLE001
                log(f"Session failed {season} R{int(round_number):02d} Q: {type(exc).__name__}: {exc}")
                log(traceback.format_exc(limit=3).strip())
                failure_rows = [
                    {
                        "lap_key": str(target["lap_key"]),
                        "season": season,
                        "round": round_number,
                        "Driver": target["Driver"],
                        "LapNumber": target["LapNumber"],
                        "status": "session_load_error",
                        "error_type": type(exc).__name__,
                        "error_msg": str(exc),
                    }
                    for _, target in group.iterrows()
                ]
                append_csv_rows(AUDIT_PATH, failure_rows)

    if AUDIT_PATH.exists():
        audit_df = pd.read_csv(AUDIT_PATH)
        success_summary_df = (
            audit_df.groupby("status", dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        success_summary_df["fraction"] = success_summary_df["count"] / len(audit_df)
        success_summary_df.to_csv(SUCCESS_PATH, index=False)

        ok_keys = set(audit_df.loc[audit_df["status"] == "ok", "lap_key"].astype(str))
        completed_targets = targets_df[targets_df["lap_key"].astype(str).isin(ok_keys)].copy()
        driver_counts_df = (
            completed_targets.groupby("Driver", dropna=False)
            .agg(n_laps=("lap_key", "count"), n_seasons=("season", "nunique"))
            .reset_index()
            .sort_values(["n_laps", "Driver"], ascending=[False, True])
        )
        driver_counts_df.to_csv(DRIVER_COUNTS_PATH, index=False)

    log("Telemetry extraction script finished")


if __name__ == "__main__":
    main()


## Source: `26_build_long_horizon_features.py`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


EXPORT_DIR = Path("exports")
TELEMETRY_PATH = EXPORT_DIR / "balanced_top6_telemetry_merged_2018_2025_strict.csv"
TARGETS_PATH = EXPORT_DIR / "balanced_top6_session_driver_best_laps_2018_2025_strict.csv"
TOP5_PATH = EXPORT_DIR / "balanced_top5_session_driver_best_laps_2018_2025_strict.csv"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def safe_frac(mask: pd.Series) -> float:
    if len(mask) == 0:
        return np.nan
    return float(mask.mean())


def first_true_frac_pos(mask: pd.Series) -> float:
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        return np.nan
    denom = max(len(mask) - 1, 1)
    return float(idx[0] / denom)


def last_true_frac_pos(mask: pd.Series) -> float:
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        return np.nan
    denom = max(len(mask) - 1, 1)
    return float(idx[-1] / denom)


def count_state_changes(mask: pd.Series) -> int:
    values = mask.astype(int).to_numpy()
    if len(values) <= 1:
        return 0
    return int(np.abs(np.diff(values)).sum())


def build_features_for_lap(lap_df: pd.DataFrame) -> dict:
    lap_df = lap_df.sort_values("sample_idx").reset_index(drop=True)

    speed = pd.to_numeric(lap_df["Speed"], errors="coerce")
    throttle = pd.to_numeric(lap_df["Throttle"], errors="coerce")
    rpm = pd.to_numeric(lap_df["RPM"], errors="coerce")
    gear = pd.to_numeric(lap_df["nGear"], errors="coerce")
    brake = (
        lap_df["Brake"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1.0, "false": 0.0})
    )
    drs = pd.to_numeric(lap_df["DRS"], errors="coerce") if "DRS" in lap_df.columns else pd.Series(np.nan, index=lap_df.index)

    throttle_zero = throttle <= 0.0
    throttle_full = throttle >= 99.0
    throttle_mid = (throttle > 0.0) & (throttle < 99.0)
    brake_active = brake >= 0.5

    speed_diff = speed.diff()
    throttle_diff = throttle.diff()
    rpm_diff = rpm.diff()
    gear_diff = gear.diff()

    features = {
        "n_samples": len(lap_df),
        "speed_mean": speed.mean(),
        "speed_std": speed.std(),
        "speed_min": speed.min(),
        "speed_max": speed.max(),
        "speed_q10": speed.quantile(0.10),
        "speed_q50": speed.quantile(0.50),
        "speed_q90": speed.quantile(0.90),
        "speed_range": speed.max() - speed.min(),
        "throttle_mean": throttle.mean(),
        "throttle_std": throttle.std(),
        "throttle_min": throttle.min(),
        "throttle_max": throttle.max(),
        "throttle_zero_frac": safe_frac(throttle_zero),
        "throttle_full_frac": safe_frac(throttle_full),
        "throttle_mid_frac": safe_frac(throttle_mid),
        "brake_mean": brake.mean(),
        "brake_std": brake.std(),
        "brake_min": brake.min(),
        "brake_max": brake.max(),
        "brake_active_frac": safe_frac(brake_active),
        "brake_on_count": count_state_changes(brake_active) // 2 + int(brake_active.iloc[0]) if len(brake_active) else 0,
        "first_brake_frac_pos": first_true_frac_pos(brake_active),
        "last_brake_frac_pos": last_true_frac_pos(brake_active),
        "rpm_mean": rpm.mean(),
        "rpm_std": rpm.std(),
        "rpm_min": rpm.min(),
        "rpm_max": rpm.max(),
        "gear_mean": gear.mean(),
        "gear_std": gear.std(),
        "gear_min": gear.min(),
        "gear_max": gear.max(),
        "gear_change_count": int(gear_diff.fillna(0).ne(0).sum()),
        "speed_diff_mean": speed_diff.mean(),
        "speed_diff_std": speed_diff.std(),
        "speed_diff_abs_mean": speed_diff.abs().mean(),
        "throttle_diff_mean": throttle_diff.mean(),
        "throttle_diff_std": throttle_diff.std(),
        "throttle_diff_abs_mean": throttle_diff.abs().mean(),
        "rpm_diff_mean": rpm_diff.mean(),
        "rpm_diff_std": rpm_diff.std(),
        "rpm_diff_abs_mean": rpm_diff.abs().mean(),
        "drs_active_frac": safe_frac(drs > 0),
    }
    return features


def main() -> None:
    telemetry_df = pd.read_csv(TELEMETRY_PATH, low_memory=False)
    targets_df = pd.read_csv(TARGETS_PATH)
    top5_df = pd.read_csv(TOP5_PATH)

    lap_features = []
    for lap_key, lap_df in telemetry_df.groupby("lap_key", sort=True):
        meta = lap_df.iloc[0]
        feature_row = build_features_for_lap(lap_df)
        feature_row.update(
            {
                "lap_key": lap_key,
                "Driver": meta["Driver"],
                "season": int(meta["season"]),
                "round": int(meta["round"]),
                "LapNumber": meta["LapNumber"],
                "LapTimeSeconds": meta.get("LapTimeSeconds"),
                "Sector1TimeSeconds": meta.get("Sector1TimeSeconds"),
                "Sector2TimeSeconds": meta.get("Sector2TimeSeconds"),
                "Sector3TimeSeconds": meta.get("Sector3TimeSeconds"),
                "Compound": meta.get("Compound"),
                "TyreLife": meta.get("TyreLife"),
                "Team": meta.get("Team"),
                "session_key": meta.get("session_key"),
            }
        )
        lap_features.append(feature_row)

    features_df = pd.DataFrame(lap_features)
    features_df = features_df.merge(
        targets_df[["lap_key", "event_name"]],
        on="lap_key",
        how="left",
    )

    export_csv(features_df, "balanced_top6_lap_features_2018_2025_strict")

    missingness_df = (
        features_df.isna().mean().reset_index()
        .rename(columns={"index": "feature", 0: "missing_fraction"})
        .sort_values(["missing_fraction", "feature"], ascending=[False, True])
    )
    export_csv(missingness_df, "balanced_top6_lap_features_missingness_2018_2025_strict")

    summary_stats_df = features_df.describe(include="all").transpose().reset_index().rename(columns={"index": "feature"})
    export_csv(summary_stats_df, "balanced_top6_lap_features_summary_stats_2018_2025_strict")

    class_balance_df = (
        features_df.groupby("Driver", dropna=False)
        .size()
        .reset_index(name="n_laps")
        .sort_values(["n_laps", "Driver"], ascending=[False, True])
    )
    export_csv(class_balance_df, "balanced_top6_lap_features_class_balance_2018_2025_strict")

    top5_keys = set(top5_df["lap_key"].astype(str))
    top5_features_df = features_df[features_df["lap_key"].astype(str).isin(top5_keys)].copy()
    export_csv(top5_features_df, "balanced_top5_lap_features_2018_2025_strict")

    top5_class_balance_df = (
        top5_features_df.groupby("Driver", dropna=False)
        .size()
        .reset_index(name="n_laps")
        .sort_values(["n_laps", "Driver"], ascending=[False, True])
    )
    export_csv(top5_class_balance_df, "balanced_top5_lap_features_class_balance_2018_2025_strict")

    print("Built long-horizon handcrafted feature tables.")
    print(class_balance_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `27_long_horizon_baseline.py`


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def has_xgboost() -> bool:
    return importlib.util.find_spec("xgboost") is not None


def build_model(model_name: str, numeric_features: list[str], categorical_features: list[str]):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    if model_name == "logistic_regression":
        estimator = LogisticRegression(
            penalty="l1",
            C=2.0,
            solver="saga",
            max_iter=5000,
            random_state=RANDOM_STATE,
            multi_class="multinomial",
        )
    elif model_name == "random_forest":
        estimator = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "hist_gradient_boosting":
        estimator = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=300,
            random_state=RANDOM_STATE,
        )
    elif model_name == "xgboost":
        from xgboost import XGBClassifier

        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    else:
        raise ValueError(model_name)

    return Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])


def run_experiment(df: pd.DataFrame, subset_name: str, experiment_name: str, model_names: list[str], feature_cols: list[str]):
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df["Driver"])
    groups = df["session_key"].astype(str).to_numpy()

    X = df[feature_cols].copy()
    categorical_features = [c for c in feature_cols if X[c].dtype == "object"]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    fold_rows = []
    summary_rows = []

    for model_name in model_names:
        oof_pred = np.full(len(df), -1, dtype=int)

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):
            x_train = X.iloc[train_idx]
            x_test = X.iloc[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]

            model = build_model(model_name, numeric_features, categorical_features)
            if model_name == "xgboost":
                model.fit(x_train, y_train)
            else:
                model.fit(x_train, y_train)

            train_pred = model.predict(x_train)
            test_pred = model.predict(x_test)
            oof_pred[test_idx] = test_pred

            fold_rows.append(
                {
                    "subset_name": subset_name,
                    "experiment": experiment_name,
                    "model": model_name,
                    "fold": fold,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "train_accuracy": accuracy_score(y_train, train_pred),
                    "test_accuracy": accuracy_score(y_test, test_pred),
                    "train_balanced_accuracy": balanced_accuracy_score(y_train, train_pred),
                    "test_balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
                    "train_macro_f1": f1_score(y_train, train_pred, average="macro"),
                    "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
                }
            )

        summary_rows.append(
            {
                "subset_name": subset_name,
                "experiment": experiment_name,
                "model": model_name,
                "n_samples": len(df),
                "n_drivers": df["Driver"].nunique(),
                "n_features": len(feature_cols),
                "oof_accuracy": accuracy_score(y, oof_pred),
                "oof_balanced_accuracy": balanced_accuracy_score(y, oof_pred),
                "oof_macro_f1": f1_score(y, oof_pred, average="macro"),
            }
        )

    return pd.DataFrame(fold_rows), pd.DataFrame(summary_rows)


def main() -> None:
    top5_df = pd.read_csv(EXPORT_DIR / "balanced_top5_lap_features_2018_2025_strict.csv")
    top6_df = pd.read_csv(EXPORT_DIR / "balanced_top6_lap_features_2018_2025_strict.csv")

    base_drop = [
        "lap_key", "Driver", "season", "round", "LapNumber",
        "Team", "event_name", "session_key", "Compound", "TyreLife",
    ]
    weak_drop = [
        "throttle_min", "brake_min", "drs_active_frac",
    ]
    time_drop = [
        "LapTimeSeconds", "Sector1TimeSeconds", "Sector2TimeSeconds", "Sector3TimeSeconds",
    ]

    datasets = {
        "balanced_top5": top5_df,
        "balanced_top6": top6_df,
    }
    experiments = {
        "with_time_features": [c for c in top6_df.columns if c not in base_drop + weak_drop],
        "without_time_features": [c for c in top6_df.columns if c not in base_drop + weak_drop + time_drop],
    }

    model_names = ["logistic_regression", "random_forest", "hist_gradient_boosting"]
    if has_xgboost():
        model_names.append("xgboost")

    all_fold_rows = []
    all_summary_rows = []

    for subset_name, df in datasets.items():
        for experiment_name, feature_cols in experiments.items():
            usable_cols = [c for c in feature_cols if c in df.columns]
            fold_df, summary_df = run_experiment(df, subset_name, experiment_name, model_names, usable_cols)
            all_fold_rows.append(fold_df)
            all_summary_rows.append(summary_df)

    fold_metrics_df = pd.concat(all_fold_rows, ignore_index=True)
    summary_df = pd.concat(all_summary_rows, ignore_index=True)

    grouped_summary_df = (
        fold_metrics_df.groupby(["subset_name", "experiment", "model"], dropna=False)
        .agg(
            mean_train_accuracy=("train_accuracy", "mean"),
            mean_test_accuracy=("test_accuracy", "mean"),
            mean_train_macro_f1=("train_macro_f1", "mean"),
            mean_test_macro_f1=("test_macro_f1", "mean"),
        )
        .reset_index()
    )
    grouped_summary_df["mean_accuracy_gap"] = grouped_summary_df["mean_train_accuracy"] - grouped_summary_df["mean_test_accuracy"]
    grouped_summary_df["mean_macro_f1_gap"] = grouped_summary_df["mean_train_macro_f1"] - grouped_summary_df["mean_test_macro_f1"]

    summary_df = summary_df.merge(grouped_summary_df, on=["subset_name", "experiment", "model"], how="left")
    summary_df = summary_df.sort_values(["subset_name", "oof_macro_f1", "oof_accuracy"], ascending=[True, False, False]).reset_index(drop=True)

    best_models_df = (
        summary_df.sort_values(["subset_name", "oof_macro_f1", "oof_accuracy"], ascending=[True, False, False])
        .groupby("subset_name", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    export_csv(fold_metrics_df, "long_horizon_baseline_fold_metrics")
    export_csv(summary_df, "long_horizon_baseline_model_summary")
    export_csv(best_models_df, "long_horizon_baseline_best_models")

    print("Long-horizon baseline modeling complete.")
    print(best_models_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `28_long_horizon_sequence_models.py`


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5
SEQ_LEN = 300
SEQ_CHANNELS = ["Speed", "Throttle", "Brake", "RPM", "nGear"]
MAX_EPOCHS = 80
BATCH_SIZE = 16
PATIENCE = 10

tf.keras.utils.set_random_seed(RANDOM_STATE)


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def resample_sequence(values: np.ndarray, target_len: int) -> np.ndarray:
    values = pd.Series(values).astype(float).interpolate(limit_direction="both").bfill().ffill()
    values = values.to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return np.zeros(target_len, dtype=float)
    if n == 1:
        return np.repeat(values[0], target_len)
    x_old = np.linspace(0.0, 1.0, n)
    x_new = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_new, x_old, values)


def standardize_sequence_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std


def standardize_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std


def build_cnn_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_hybrid_cnn_model(seq_len: int, n_channels: int, n_tab_features: int, n_classes: int) -> tf.keras.Model:
    seq_in = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    tab_in = tf.keras.Input(shape=(n_tab_features,), name="tab_input")
    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(seq_in)
    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.MaxPooling1D(2)(x_seq)
    x_seq = tf.keras.layers.Dropout(0.2)(x_seq)
    x_seq = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.GlobalAveragePooling1D()(x_seq)
    x_tab = tf.keras.layers.Dense(32, activation="relu")(tab_in)
    x_tab = tf.keras.layers.Dropout(0.2)(x_tab)
    x = tf.keras.layers.Concatenate()([x_seq, x_tab])
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=[seq_in, tab_in], outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_gru_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.GRU(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.GRU(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_lstm_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.LSTM(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def prepare_subset_data(subset_name: str):
    telemetry_df = pd.read_csv(EXPORT_DIR / "balanced_top6_telemetry_merged_2018_2025_strict.csv", low_memory=False)
    features_path = EXPORT_DIR / f"{subset_name}_lap_features_2018_2025_strict.csv"
    features_df = pd.read_csv(features_path)

    drivers = sorted(features_df["Driver"].unique().tolist())
    telemetry_df = telemetry_df[telemetry_df["Driver"].isin(drivers)].copy()
    telemetry_df["Brake"] = telemetry_df["Brake"].astype(str).str.lower().map({"true": 1.0, "false": 0.0})

    base_drop = [
        "lap_key", "Driver", "season", "round", "LapNumber",
        "Team", "event_name", "session_key", "Compound", "TyreLife",
        "throttle_min", "brake_min", "drs_active_frac",
    ]
    tab_features = [c for c in features_df.columns if c not in base_drop]

    lap_rows = []
    sequence_arrays = []
    for lap_key, lap_df in telemetry_df.sort_values(["lap_key", "sample_idx"]).groupby("lap_key"):
        lap_meta = lap_df.iloc[0]
        channel_matrix = []
        for channel in SEQ_CHANNELS:
            series = pd.to_numeric(lap_df[channel], errors="coerce")
            channel_matrix.append(resample_sequence(series.values, SEQ_LEN))
        sequence_arrays.append(np.stack(channel_matrix, axis=-1))
        lap_rows.append(
            {
                "lap_key": lap_key,
                "Driver": lap_meta["Driver"],
                "season": int(lap_meta["season"]),
                "round": int(lap_meta["round"]),
                "session_key": lap_meta["session_key"],
                "raw_len": len(lap_df),
            }
        )

    sequence_meta_df = pd.DataFrame(lap_rows)
    meta_df = sequence_meta_df.merge(features_df[["lap_key"] + tab_features], on="lap_key", how="inner")
    sequence_lookup = {row["lap_key"]: idx for idx, row in sequence_meta_df.reset_index().iterrows()}
    aligned_indices = meta_df["lap_key"].map(sequence_lookup).to_numpy()
    X_seq = np.stack(sequence_arrays)[aligned_indices].astype(np.float32)
    X_tab = meta_df[tab_features].astype(float).to_numpy(dtype=np.float32)
    y_labels = meta_df["Driver"].astype(str).to_numpy()
    groups = meta_df["session_key"].astype(str).to_numpy()

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_labels)

    return {
        "subset_name": subset_name,
        "drivers": drivers,
        "tab_features": tab_features,
        "meta_df": meta_df,
        "X_seq": X_seq,
        "X_tab": X_tab,
        "y": y,
        "y_labels": y_labels,
        "groups": groups,
        "label_encoder": label_encoder,
    }


def run_subset_models(data_bundle: dict, handcrafted_summary: pd.DataFrame):
    subset_name = data_bundle["subset_name"]
    X_seq = data_bundle["X_seq"]
    X_tab = data_bundle["X_tab"]
    y = data_bundle["y"]
    groups = data_bundle["groups"]
    n_classes = len(np.unique(y))

    outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    outer_splits = []
    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_seq, y, groups), start=1):
        inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + fold_idx)
        train_inner_rel, val_rel = next(inner_cv.split(X_seq[train_idx], y[train_idx], groups[train_idx]))
        outer_splits.append(
            {
                "fold": fold_idx,
                "train_idx": train_idx,
                "train_inner_idx": train_idx[train_inner_rel],
                "val_idx": train_idx[val_rel],
                "test_idx": test_idx,
            }
        )

    split_rows = [
        {
            "subset_name": subset_name,
            "fold": split["fold"],
            "n_train": len(split["train_idx"]),
            "n_train_inner": len(split["train_inner_idx"]),
            "n_val": len(split["val_idx"]),
            "n_test": len(split["test_idx"]),
        }
        for split in outer_splits
    ]

    model_defs = {
        "cnn": lambda: build_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
        "gru": lambda: build_gru_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
        "lstm": lambda: build_lstm_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
        "hybrid_cnn_tabular": lambda: build_hybrid_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), X_tab.shape[1], n_classes),
    }

    fold_rows = []
    history_rows = []
    summary_rows = []

    for model_name, builder in model_defs.items():
        oof_pred = np.full(len(y), -1, dtype=int)

        for split in outer_splits:
            train_idx = split["train_idx"]
            train_inner_idx = split["train_inner_idx"]
            val_idx = split["val_idx"]
            test_idx = split["test_idx"]

            seq_train_inner, seq_val, seq_test = standardize_sequence_by_train(
                X_seq[train_inner_idx], X_seq[val_idx], X_seq[test_idx]
            )
            seq_train_full, _, _ = standardize_sequence_by_train(
                X_seq[train_idx], X_seq[val_idx], X_seq[test_idx]
            )
            tab_train_inner, tab_val, tab_test = standardize_by_train(
                X_tab[train_inner_idx], X_tab[val_idx], X_tab[test_idx]
            )
            tab_train_full, _, _ = standardize_by_train(
                X_tab[train_idx], X_tab[val_idx], X_tab[test_idx]
            )

            y_train_inner = y[train_inner_idx]
            y_val = y[val_idx]
            y_train_full = y[train_idx]
            y_test = y[test_idx]

            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=PATIENCE,
                    restore_best_weights=True,
                )
            ]

            model = builder()
            if model_name == "hybrid_cnn_tabular":
                history = model.fit(
                    [seq_train_inner, tab_train_inner],
                    y_train_inner,
                    validation_data=([seq_val, tab_val], y_val),
                    epochs=MAX_EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    callbacks=callbacks,
                )
                train_pred = np.argmax(model.predict([seq_train_full, tab_train_full], verbose=0), axis=1)
                test_pred = np.argmax(model.predict([seq_test, tab_test], verbose=0), axis=1)
            else:
                history = model.fit(
                    seq_train_inner,
                    y_train_inner,
                    validation_data=(seq_val, y_val),
                    epochs=MAX_EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    callbacks=callbacks,
                )
                train_pred = np.argmax(model.predict(seq_train_full, verbose=0), axis=1)
                test_pred = np.argmax(model.predict(seq_test, verbose=0), axis=1)

            oof_pred[test_idx] = test_pred

            history_rows.append(
                {
                    "subset_name": subset_name,
                    "model": model_name,
                    "fold": split["fold"],
                    "epochs_trained": len(history.history["loss"]),
                    "best_val_loss": float(np.min(history.history["val_loss"])),
                }
            )

            fold_rows.append(
                {
                    "subset_name": subset_name,
                    "model": model_name,
                    "fold": split["fold"],
                    "train_accuracy": accuracy_score(y_train_full, train_pred),
                    "test_accuracy": accuracy_score(y_test, test_pred),
                    "train_balanced_accuracy": balanced_accuracy_score(y_train_full, train_pred),
                    "test_balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
                    "train_macro_f1": f1_score(y_train_full, train_pred, average="macro"),
                    "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
                }
            )

        summary_rows.append(
            {
                "subset_name": subset_name,
                "model": model_name,
                "n_samples": len(y),
                "n_drivers": n_classes,
                "oof_accuracy": accuracy_score(y, oof_pred),
                "oof_balanced_accuracy": balanced_accuracy_score(y, oof_pred),
                "oof_macro_f1": f1_score(y, oof_pred, average="macro"),
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    grouped = (
        fold_df.groupby(["subset_name", "model"], dropna=False)
        .agg(
            mean_train_accuracy=("train_accuracy", "mean"),
            mean_test_accuracy=("test_accuracy", "mean"),
            mean_train_macro_f1=("train_macro_f1", "mean"),
            mean_test_macro_f1=("test_macro_f1", "mean"),
        )
        .reset_index()
    )
    grouped["mean_accuracy_gap"] = grouped["mean_train_accuracy"] - grouped["mean_test_accuracy"]
    grouped["mean_macro_f1_gap"] = grouped["mean_train_macro_f1"] - grouped["mean_test_macro_f1"]

    summary_df = pd.DataFrame(summary_rows).merge(grouped, on=["subset_name", "model"], how="left")
    handcrafted_best = handcrafted_summary[
        (handcrafted_summary["subset_name"] == subset_name) &
        (handcrafted_summary["model"] == "logistic_regression")
    ].sort_values(["oof_macro_f1", "oof_accuracy"], ascending=[False, False]).head(1)

    compare_rows = []
    if not handcrafted_best.empty:
        base_acc = handcrafted_best["oof_accuracy"].iloc[0]
        base_f1 = handcrafted_best["oof_macro_f1"].iloc[0]
        compare_rows.append(
            {
                "subset_name": subset_name,
                "model_family": "handcrafted_compact_logreg",
                "oof_accuracy": base_acc,
                "oof_macro_f1": base_f1,
                "accuracy_vs_handcrafted": 0.0,
                "macro_f1_vs_handcrafted": 0.0,
            }
        )
        for _, row in summary_df.iterrows():
            compare_rows.append(
                {
                    "subset_name": subset_name,
                    "model_family": row["model"],
                    "oof_accuracy": row["oof_accuracy"],
                    "oof_macro_f1": row["oof_macro_f1"],
                    "accuracy_vs_handcrafted": row["oof_accuracy"] - base_acc,
                    "macro_f1_vs_handcrafted": row["oof_macro_f1"] - base_f1,
                }
            )

    return (
        pd.DataFrame(split_rows),
        fold_df,
        pd.DataFrame(history_rows),
        summary_df.sort_values(["subset_name", "oof_macro_f1"], ascending=[True, False]).reset_index(drop=True),
        pd.DataFrame(compare_rows),
    )


def main():
    handcrafted_summary = pd.read_csv(EXPORT_DIR / "long_horizon_baseline_model_summary.csv")

    all_split = []
    all_fold = []
    all_history = []
    all_summary = []
    all_compare = []

    for subset_name in ["balanced_top5", "balanced_top6"]:
        bundle = prepare_subset_data(subset_name)
        split_df, fold_df, history_df, summary_df, compare_df = run_subset_models(bundle, handcrafted_summary)
        all_split.append(split_df)
        all_fold.append(fold_df)
        all_history.append(history_df)
        all_summary.append(summary_df)
        all_compare.append(compare_df)

    split_df = pd.concat(all_split, ignore_index=True)
    fold_df = pd.concat(all_fold, ignore_index=True)
    history_df = pd.concat(all_history, ignore_index=True)
    summary_df = pd.concat(all_summary, ignore_index=True)
    compare_df = pd.concat(all_compare, ignore_index=True)

    best_df = (
        summary_df.sort_values(["subset_name", "oof_macro_f1", "oof_accuracy"], ascending=[True, False, False])
        .groupby("subset_name", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    export_csv(split_df, "long_horizon_sequence_split_summary")
    export_csv(fold_df, "long_horizon_sequence_fold_metrics")
    export_csv(history_df, "long_horizon_sequence_training_history")
    export_csv(summary_df, "long_horizon_sequence_model_summary")
    export_csv(best_df, "long_horizon_sequence_best_models")
    export_csv(compare_df, "long_horizon_sequence_vs_handcrafted")

    print("Long-horizon sequence modeling complete.")
    print(best_df.to_string(index=False))


if __name__ == "__main__":
    main()
